# Test the deployed tire calculator

## Goal

Open the published calculator in Chromium and verify that the Python model
loads, renders ten ranked tire pairings and recalculates after an input change.

In [1]:
import os

from playwright.async_api import async_playwright

## Setup

The deployment workflow supplies the exact GitHub Pages URL and commit query.

In [2]:
calculator_url = os.environ["CALCULATOR_URL"]

## Test

The browser must reach the ready state without page errors. The test then
changes system weight from 198 lb to 265 lb and requires the displayed front
pressure to change.

In [3]:
browser_errors = []

async with async_playwright() as playwright:
    browser = await playwright.chromium.launch()
    page = await browser.new_page()
    page.on(
        "pageerror",
        lambda error: browser_errors.append(str(error)),
    )

    response = await page.goto(
        calculator_url,
        wait_until="domcontentloaded",
        timeout=120_000,
    )
    assert response is not None and response.ok
    await page.locator('body[data-ready="true"]').wait_for(
        timeout=120_000
    )
    assert "mph" in await page.locator(
        'label:has(#speed-from)'
    ).inner_text()
    assert await page.locator("#speed-from").input_value() == "21"
    assert await page.locator("#speed-to").input_value() == "27"
    assert await page.get_by_text("Wind exposure").count() == 0
    model_note = await page.locator(".model-note").inner_text()
    assert "not a 105% rule" in model_note
    assert "29.8 and 31.4 mm" in model_note

    winner_card = page.locator('[data-testid="winner-card"]')
    await winner_card.wait_for()
    ranking_rows = page.locator("#ranking-table tbody tr")
    assert await ranking_rows.count() == 10

    pressure_text_before = await winner_card.locator("p").first.text_content()
    await page.locator("#system-weight").fill("265")
    await page.locator("#calculate-button").click()
    await page.wait_for_function(
        """previousPressure => {
            const pressure = document.querySelector(
                '[data-testid="winner-card"] p'
            );
            return pressure && pressure.textContent !== previousPressure;
        }""",
        arg=pressure_text_before,
        timeout=30_000,
    )
    pressure_text_after = await winner_card.locator("p").first.text_content()

    assert pressure_text_before != pressure_text_after
    assert await page.locator(
        '#runtime-status[data-state="error"]'
    ).count() == 0
    assert browser_errors == []

    winner_text = await winner_card.inner_text()
    await browser.close()

{
    "url": calculator_url,
    "winner": winner_text,
    "ranking_rows": 10,
    "pressure_before": pressure_text_before,
    "pressure_after": pressure_text_after,
    "browser_errors": browser_errors,
}

{'url': 'http://127.0.0.1:8765/',
 'winner': 'FASTEST MODELED SYSTEM\nFRONT\nSL-R 28\n\n77.6 psi · 29.5 mm mounted\n\nREAR\nTT TR 28\n\n78.9 psi · 29.2 mm mounted\n\n32.8 W modeled tire-system loss',
 'ranking_rows': 10,
 'pressure_before': '73.4 psi · 29.5 mm mounted',
 'pressure_after': '77.6 psi · 29.5 mm mounted',
 'browser_errors': []}

## Result

A completed notebook is evidence that the deployed browser runtime, Python
model, result rendering and recalculation path all worked together.